# UIT_SGS_Foodseekers_C1 — XLM-RoBERTa **v2 (cải thiện)** · Lượt 4
Mục tiêu: vượt **0.95082** (Lượt 3).

**Cải tiến dựa trên nghiên cứu (kaggle NLP gold-medal playbook):**
1. **5-Fold Cross-Validation Ensemble** — train 5 model trên 5 fold, lấy trung bình xác suất test. Đây là đòn tăng Macro F1 ổn định và mạnh nhất cho Transformer.
2. **Multi-Sample Dropout** ở đầu phân loại — giảm overfit, tăng khái quát hoá.
3. **max_len 320 + 4 epoch + warmup + weighted CE loss** (chống mất cân bằng 62/31/7).
4. **Tinh chỉnh ngưỡng lớp hiếm trên OOF** (gộp 5 fold) — ổn định hơn chỉnh trên 1 tập val.

Chạy: **Runtime → T4 GPU → Run all**. Lâu hơn Lượt 3 (~5 lần, khoảng 60–90 phút trên T4).

In [ ]:
!pip -q install -U transformers accelerate
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHONG CO GPU - bat T4!")

In [ ]:
# Cấu hình
MODEL_NAME = "FacebookAI/xlm-roberta-base"
MAX_LEN, EPOCHS, LR, BATCH, SEED, N_FOLDS = 320, 4, 2e-5, 16, 42, 5
BASE_URL = "https://raw.githubusercontent.com/bersious/Foodseekers/main/data"
LABELS = ["BENIGN", "JAILBREAK", "PROMPT_INJECTION"]; L2I = {l:i for i,l in enumerate(LABELS)}
import random, numpy as np
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
import pandas as pd
train = pd.read_csv(f"{BASE_URL}/train.csv"); test = pd.read_csv(f"{BASE_URL}/test.csv")
train["text"] = train["text"].fillna("").astype(str); test["text"] = test["text"].fillna("").astype(str)
y_all = train["label"].map(L2I).values
print(train.shape, test.shape, {LABELS[i]: int((y_all==i).sum()) for i in range(3)})

In [ ]:
# Tokenizer + Dataset (padding động)
from transformers import AutoTokenizer
from torch.utils.data import Dataset, DataLoader
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
class DS(Dataset):
    def __init__(self, texts, labels=None): self.texts=list(texts); self.labels=labels
    def __len__(self): return len(self.texts)
    def __getitem__(self, i): return (self.texts[i], -1 if self.labels is None else int(self.labels[i]))
def collate(b):
    t,l = zip(*b)
    enc = tok(list(t), truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt")
    enc["labels"] = torch.tensor(l); return enc

In [ ]:
# Model: XLM-R + đầu phân loại Multi-Sample Dropout
import torch.nn as nn
from transformers import AutoModel

class MSDClassifier(nn.Module):
    def __init__(self, name, n_cls=3, n_drop=5):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(name)
        h = self.backbone.config.hidden_size
        self.drops = nn.ModuleList([nn.Dropout(0.3) for _ in range(n_drop)])
        self.fc = nn.Linear(h, n_cls)
    def forward(self, input_ids=None, attention_mask=None, **kw):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        pooled = out.last_hidden_state[:, 0]          # CLS token
        logits = sum(self.fc(d(pooled)) for d in self.drops) / len(self.drops)
        return logits

In [ ]:
# Huấn luyện 1 fold -> trả xác suất val (OOF) + xác suất test
from transformers import get_linear_schedule_with_warmup
from sklearn.metrics import f1_score
from scipy.special import softmax
import copy

def run_fold(tr_idx, va_idx, class_w):
    dl_tr = DataLoader(DS(train["text"].iloc[tr_idx], y_all[tr_idx]), batch_size=BATCH, shuffle=True, collate_fn=collate)
    dl_va = DataLoader(DS(train["text"].iloc[va_idx], y_all[va_idx]), batch_size=BATCH*2, collate_fn=collate)
    dl_te = DataLoader(DS(test["text"]), batch_size=BATCH*2, collate_fn=collate)
    model = MSDClassifier(MODEL_NAME).to(DEVICE)
    loss_fn = nn.CrossEntropyLoss(weight=class_w.to(DEVICE))
    optim = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    steps = len(dl_tr)*EPOCHS
    sched = get_linear_schedule_with_warmup(optim, int(0.1*steps), steps)
    scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

    @torch.no_grad()
    def infer(dl):
        model.eval(); out=[]
        for enc in dl:
            enc.pop("labels"); enc={k:v.to(DEVICE) for k,v in enc.items()}
            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                out.append(model(**enc).float().cpu().numpy())
        return np.concatenate(out)

    best_f1, best_state = -1, None
    for ep in range(EPOCHS):
        model.train()
        for enc in dl_tr:
            lb = enc.pop("labels").to(DEVICE); enc={k:v.to(DEVICE) for k,v in enc.items()}
            optim.zero_grad()
            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                loss = loss_fn(model(**enc), lb)
            scaler.scale(loss).backward(); scaler.step(optim); scaler.update(); sched.step()
        f = f1_score(y_all[va_idx], infer(dl_va).argmax(1), average="macro")
        print(f"    epoch {ep+1}: val macro F1 = {f:.4f}")
        if f > best_f1: best_f1, best_state = f, copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    return softmax(infer(dl_va),1), softmax(infer(dl_te),1), best_f1

In [ ]:
# Vòng lặp 5-Fold Ensemble
from sklearn.model_selection import StratifiedKFold
counts = np.bincount(y_all, minlength=3)
class_w = torch.tensor(len(y_all)/(3*counts), dtype=torch.float)
print("Class weights:", class_w.numpy().round(3))

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
oof = np.zeros((len(train),3)); test_prob = np.zeros((len(test),3)); fold_f1=[]
for k,(tr_idx,va_idx) in enumerate(skf.split(train, y_all)):
    print(f"\n===== FOLD {k+1}/{N_FOLDS} =====")
    p_va, p_te, bf = run_fold(tr_idx, va_idx, class_w)
    oof[va_idx] = p_va; test_prob += p_te/N_FOLDS; fold_f1.append(bf)
print("\nFold F1:", [round(x,4) for x in fold_f1], "| mean:", round(np.mean(fold_f1),4))
print("OOF Macro F1 (argmax):", round(f1_score(y_all, oof.argmax(1), average="macro"),4))

In [ ]:
# Tinh chỉnh ngưỡng lớp hiếm trên OOF (ổn định, dùng toàn 14k mẫu)
best_w, best = (1.0,1.0), -1
for wj in np.arange(0.8,1.31,0.05):
    for wp in np.arange(0.8,2.01,0.05):
        f = f1_score(y_all,(oof*[1.0,wj,wp]).argmax(1),average="macro")
        if f>best: best,best_w = f,(round(float(wj),2),round(float(wp),2))
print(f"He so JAILBREAK x{best_w[0]}, PROMPT_INJECTION x{best_w[1]} -> OOF Macro F1 = {best:.4f}")
from sklearn.metrics import classification_report
print(classification_report(y_all,(oof*[1.0,*best_w]).argmax(1),target_names=LABELS,digits=4))

In [ ]:
# Dự đoán test + kiểm tra + lưu
pred = (test_prob*[1.0,*best_w]).argmax(1)
sub = pd.DataFrame({"id": test["id"], "label":[LABELS[i] for i in pred]})
assert len(sub)==6000 and sub["label"].isin(LABELS).all() and sub["label"].notna().all() and sub["id"].is_unique
sub.to_csv("submission.csv", index=False)
np.save("xlmr_5fold_test_proba.npy", test_prob)
np.save("xlmr_5fold_oof_proba.npy", oof)
print(sub["label"].value_counts().to_dict())

In [ ]:
try:
    from google.colab import files
    files.download("submission.csv"); files.download("xlmr_5fold_test_proba.npy")
except Exception: print("File o thu muc hien tai.")